# Solutions · Chapter 03-02 · Why two samples never agree

E8 and E15 are the two worth attempting before reading - both have results that are mildly alarming
and change how you read other people's numbers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pop_rng = np.random.default_rng(11)
population = pop_rng.gamma(2.0, 3.0, 1_000_000)

cont_rng = np.random.default_rng(5)
size = 1_000_000
ordinary = cont_rng.gamma(2.0, 3.0, size)
breakdown = cont_rng.random(size) < 0.03
contaminated = np.where(breakdown, ordinary + cont_rng.gamma(2.0, 60.0, size), ordinary)

print("population   : mean %.3f  sd %.3f" % (population.mean(), population.std()))
print("contaminated : mean %.3f  sd %.3f" % (contaminated.mean(), contaminated.std()))

## E1 · Two spreads

**Standard deviation:** how far individual observations sit from their mean. A property of the data,
and of the population behind it.

**Standard error:** how far your *computed summary* would sit from the summary you would get on a
different sample. A property of your estimate, and it depends on the sample size.

## E2 · Quadruple the sample

**Standard error: halves** - it goes as `sd / sqrt(n)`, and `sqrt(4) = 2`.

**Sample standard deviation: essentially unchanged** - it converges on the population's standard
deviation, which is a fixed number. It becomes a *better estimate* of that number, which is not the
same as becoming smaller.

## E3 · Why the sampling distribution is narrower and more symmetric

**Narrower** because averaging cancels. A sample containing an unusually late morning probably also
contains ordinary ones, so extremes get diluted by everything else in the sample - and the more
values you average, the more thorough the dilution.

**More symmetric** because the skew is carried by rare large values, and it takes an improbable
coincidence for a sample to be made mostly of them. A single draw is late 30 minutes reasonably
often; a sample of seven averages 30 minutes almost never. That asymmetry between "one extreme" and
"seven extremes at once" is what pulls the sampling distribution towards symmetry, and it is the
central limit theorem in one sentence.

## E4 · Sample sizes for a target standard error

In [ ]:
for target in [2.0, 1.0, 0.5]:
    print("sd 20, standard error %.1f  ->  n = (20 / %.1f)^2 = %.0f"
          % (target, target, (20 / target) ** 2))

100, 400 and 1,600. Each halving of the standard error costs **four times** the data.

**The pattern is called diminishing returns** - specifically the inverse-square-root law. The first
100 observations buy you a standard error of 2; the next 1,500 buy you the last 0.5.

## E5 · Survey A (400) versus survey B (1,600)

**B's interval is twice as narrow**, not four times: `sqrt(1600 / 400) = 2`.

**What is wrong with "four times better":** it confuses the sample size ratio with the precision
ratio. B costs four times as much to run and delivers a factor of two. Whether that is worth it is a
real question, and stating it as "four times better" answers it dishonestly - it is the exact
argument someone makes when they want the bigger budget approved.

## E6 · The formula against the simulation

In [ ]:
def standard_error(pop, n, repeats=20_000, rng=None):
    rng = rng or np.random.default_rng(0)
    draws = pop[rng.integers(0, len(pop), (repeats, n))]
    return float(draws.mean(axis=1).std())


sizes = np.array([2, 3, 5, 8, 12, 20, 35, 60, 100, 200])
rng = np.random.default_rng(0)
simulated = np.array([standard_error(population, int(n), rng=rng) for n in sizes])
formula = population.std() / np.sqrt(sizes)

print(pd.DataFrame({"n": sizes, "simulated": simulated.round(4),
                    "sd / sqrt(n)": formula.round(4),
                    "ratio": (simulated / formula).round(4)}).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(sizes, simulated, "o", color="#0072B2", label="simulated")
ax.plot(sizes, formula, "-", color="#D55E00", label="sd / sqrt(n)")
ax.set_xlabel("sample size n")
ax.set_ylabel("standard error of the mean")
ax.set_title("The formula is not an approximation")
ax.legend()
plt.tight_layout()
plt.show()

The ratios sit within 1% of 1.00 across two orders of magnitude, including at n = 2. **The `sd / sqrt(n)`
formula is exact for the standard error of a mean** - it does not depend on the population's shape,
and it is not a large-sample approximation. That is worth separating clearly from the central limit
theorem, which *is* an approximation and *does* need `n` to be large: the formula gives you the right
**width** at any `n`; the CLT tells you when the **shape** is close to normal.

## E7 · Comparing medians instead of means

In [ ]:
compare_rng = np.random.default_rng(9)

rows = []
for label, pop in [("moderately skewed", population), ("contaminated", contaminated)]:
    a = pop[compare_rng.integers(0, len(pop), (40_000, 100))]
    b = pop[compare_rng.integers(0, len(pop), (40_000, 100))]
    for statistic, apply in [("mean", lambda x: x.mean(axis=1)),
                             ("median", lambda x: np.median(x, axis=1))]:
        va, vb = apply(a), apply(b)
        relative = 100 * np.abs(vb - va) / va
        rows.append({"population": label, "compared using": statistic,
                     "typical apparent difference": "%.2f%%" % np.median(relative),
                     "share above 5%": round(float((relative > 5).mean()), 3),
                     "share above 10%": round(float((relative > 10).mean()), 3)})

print(pd.DataFrame(rows).to_string(index=False))

**On the contaminated population the false-winner problem gets dramatically better with medians:** the
typical apparent difference between two identical arms falls from **24.58% to 9.13%**, and the share
exceeding 10% falls from 0.788 to 0.458.

**On the moderately skewed population it gets worse:** 6.72% becomes 9.03%.

This is the chapter's last section restated as a decision. The noise in a comparison is driven by the
standard error of whatever statistic you compare, and the median's standard error beats the mean's
only when the tails are heavy. On the contaminated data, comparing means is close to hopeless at
n = 100 - a fifth of trials show an apparent 25% difference from nothing at all - because a single
breakdown morning landing in one arm moves that arm's mean by more than any real effect would.

**The practical rule:** if your outcome has occasional enormous values, comparing means needs either
a robust statistic or far more data than you think. Note that both remain bad in absolute terms -
71.4% of contaminated median-comparisons still exceed 5%. Robustness improved the situation; it did
not solve it.

## E8 · Estimating the standard error from the sample you have

In [ ]:
self_rng = np.random.default_rng(4)
draws = population[self_rng.integers(0, len(population), (60_000, 7))]

true_se = float(draws.mean(axis=1).std())
estimates = draws.std(axis=1, ddof=1) / np.sqrt(7)

print("true standard error at n = 7            : %.4f" % true_se)
print("average of the per-sample estimates     : %.4f" % estimates.mean())
print("bias                                    : %.4f  (%.1f%%)"
      % (estimates.mean() - true_se, 100 * (estimates.mean() / true_se - 1)))
print()
print("share of samples that UNDERSTATE their own uncertainty: %.3f"
      % (estimates < true_se).mean())
print("worst 5%% of estimates are below         : %.4f" % np.quantile(estimates, 0.05))

### The direction is the point

The per-sample estimate averages **1.4878** against a true standard error of **1.6086** - it is
**biased low by 7.5%**, and **64.2% of samples understate their own uncertainty**.

So on small samples, the error bar you can compute is systematically too narrow, and more often than
not it is too narrow for *your particular sample*. Two causes, both from earlier work:

- the sample standard deviation is biased low on small samples (03-02's footnote - `n - 1` fixes the
  variance, not its square root);
- an unlucky sample that happens to miss the tail looks *both* wrong and confident, because the same
  missing extreme values that bias the mean also shrink the estimated spread.

**What to do about it:** at small `n`, treat any computed interval as optimistic, and prefer methods
that do not lean on a single sample's standard deviation. That is one of the reasons 03-03 uses the
bootstrap, and why the `t` distribution exists - it widens intervals by exactly the amount that this
bias costs you.

## E9 · The conversion dashboard

In [ ]:
p, sessions = 0.037, 200
week_se = np.sqrt(p * (1 - p) / sessions)

print("a conversion rate is a mean of zeros and ones, so its sd is sqrt(p(1-p))")
print("standard error of one week          : %.2f percentage points" % (100 * week_se))
print("standard error of a week-to-week gap: %.2f percentage points" % (100 * week_se * np.sqrt(2)))
print()
print("the dashboard's swing, 3.1% to 4.4%, is 1.3 percentage points")
print("that is %.2f standard errors of a difference" % (1.3 / (100 * week_se * np.sqrt(2))))
print()
sim = np.random.default_rng(3).binomial(sessions, p, (20_000, 8)) / sessions
spans = 100 * (sim.max(axis=1) - sim.min(axis=1))
print("simulating eight weeks where the true rate NEVER changes:")
print("  typical span from best to worst week : %.2f percentage points" % spans.mean())
print("  share of eight-week runs with a span above 1.3 points: %.3f" % (spans > 1.3).mean())

**Answer: none of them.**

One week of 200 sessions has a standard error of **1.33 percentage points**, and a week-to-week
difference has one of **1.89**. The entire eight-week range - 3.1% to 4.4% - is 1.3 points, which is
**0.69 of a single standard error**.

Put the other way round: if the true rate never moved at all, eight weeks of this dashboard would
typically span **3.76 points**, and **99.8% of eight-week runs would show a span wider than the one
that prompted the question**. The dashboard is not merely explicable by noise; it is *quieter* than
noise, which if anything suggests the sessions are less independent than assumed.

**What to say:** "Each week's number is worth plus or minus about 1.3 points, so the whole range we
are looking at is inside the noise. To detect a change of one point we would need roughly 3,000
sessions a week - or we can pool four weeks at a time and watch that instead."

## E10 · Twelve patients

In [ ]:
print("mean improvement 8 points, sd 14, n = 12")
print("standard error = 14 / sqrt(12) = %.2f points" % (14 / np.sqrt(12)))
print("roughly, plus or minus 2 standard errors: %.1f to %.1f"
      % (8 - 2 * 14 / np.sqrt(12), 8 + 2 * 14 / np.sqrt(12)))

The standard error is **4.04 points** on an estimate of 8, so a rough interval runs from about
**-0.1 to +16.1**. The study is consistent with an improvement of 16 points and also with no
improvement at all.

**What to say to someone reporting it as "an 8-point improvement":** "8 is the best single guess, and
with twelve patients it is worth plus or minus about 8. The study cannot distinguish a large effect
from none, which is not a criticism of the result - it is what twelve patients buys. What it can do
is justify a larger trial, and it tells us roughly how large: to get the interval down to plus or
minus 2 points we would need about 200 patients."

(That last figure is `(14 / 1) ** 2` for a standard error of 1, i.e. 196 - the same arithmetic as E4.)

## E11 · "Mean 4.2 minutes, standard error 3.9 minutes, n = 2,000"

In [ ]:
print("if 3.9 is really the standard DEVIATION, then the standard error is")
print("  3.9 / sqrt(2000) = %.4f minutes" % (3.9 / np.sqrt(2000)))

**Error one: that number is a standard deviation, not a standard error.** With 2,000 sessions the
standard error would be `3.9 / sqrt(2000)` = **0.087 minutes**, about five seconds. A standard error
nearly as large as the mean at n = 2,000 would require a standard deviation of 174 minutes.

**Error two: the conclusion is backwards.** Even taken at face value, a large spread of *individual
sessions* says nothing about the reliability of the *estimate* - and the correct standard error here
makes this one of the more precisely known numbers you will meet. The analyst has read "sessions vary
a lot" as "I do not know the average", which is the sd-versus-se confusion from the chapter's first
failure lab, with the practical consequence of discarding a perfectly good result.

**What the reported number does tell you:** sessions vary enormously relative to their mean - 3.9
against 4.2 - so the distribution is heavily right-skewed and the *mean session length* is probably
not the summary anyone wants. That is a real finding hiding inside a mistaken one.

## E12 · Is a 3% improvement real?

> "First I would ask how large the arms are, because 3% means nothing without it - two identical
> groups of 100 differ by more than 5% most of the time. Then I would work out what the noise alone
> produces at that size: either from the standard error of each arm, remembering that a difference
> carries the uncertainty of both, or by simulating two arms drawn from the same pool and seeing how
> often they differ by 3%. If a 3% gap is common under no effect, the test has not shown anything and
> I would say what sample size *would* settle it. I would also check that the split was random and
> that nothing else differs between the arms, since a real difference in the wrong thing is worse
> than noise. And I would ask what a 3% improvement is worth, because the sample size needed to
> detect it reliably may cost more than the improvement does."

Five sentences, no significance, and the last one is the sentence that separates an analyst from a
calculator.

## E13 · How much would 0.83 accuracy move?

In [ ]:
accuracy = 0.83
sd_of_correctness = np.sqrt(accuracy * (1 - accuracy))
print("accuracy is a mean of 0s and 1s, so sd = sqrt(0.83 x 0.17) = %.4f" % sd_of_correctness)
print("standard error on 200 test rows = %.4f = %.1f percentage points"
      % (sd_of_correctness / np.sqrt(200), 100 * sd_of_correctness / np.sqrt(200)))
print()
print("to compare two models scoring 0.83 and 0.85 - a gap of 2 points:")
for n in [200, 1_000, 5_000, 10_000]:
    se_difference = sd_of_correctness / np.sqrt(n) * np.sqrt(2)
    print("  test set %5d rows: se of the difference %.4f -> the gap is %.2f standard errors"
          % (n, se_difference, 0.02 / se_difference))

**On 200 rows, 0.83 is worth plus or minus about 2.7 percentage points** - a different test set of the
same size would plausibly return anything from about 0.78 to 0.88.

**To distinguish 0.83 from 0.85** the gap needs to be large relative to the standard error of the
difference. At 200 rows the gap is 0.53 standard errors, which is nothing. It reaches 2.66 at
**5,000 rows** and 3.76 at 10,000.

So: **a 2-point accuracy difference measured on a 200-row test set is not a difference.** This is the
single most useful transfer in the chapter, because model comparison on small test sets is
everywhere - a leaderboard, a paper, a colleague's notebook - and the standard error is almost never
quoted. Module 07 returns to this with paired comparisons, which do better by reusing the same rows
for both models and removing one source of variation entirely.

## E14 · For the transport committee

> "We compared a hundred mornings on the old timetable with a hundred on the new one, and the new one
> came out 13% better. The difficulty is that a hundred mornings is not very many, and mornings vary a
> lot on their own - roadworks, weather, school holidays. We checked what happens when you compare two
> sets of a hundred mornings that had *no* change between them at all: they differ by more than 5%
> about six times in ten, and by more than 10% about three times in ten. So a 13% gap is the sort of
> thing this trial produces whether or not the timetable does anything. To get an answer worth acting
> on we would need to run it for longer."

95 words, no jargon, and it gives the committee something to decide rather than only something to
doubt.

## E15 · The standard error of the maximum

In [ ]:
max_rng = np.random.default_rng(6)
normal_pop = max_rng.normal(0, 1, 1_000_000)

rows = []
for label, pop in [("skewed (gamma)", population), ("normal", normal_pop)]:
    for n in [10, 100, 1_000, 10_000]:
        draws = pop[max_rng.integers(0, len(pop), (20_000, n))]
        rows.append({"population": label, "n": n,
                     "se of the mean": round(float(draws.mean(axis=1).std()), 4),
                     "se of the maximum": round(float(draws.max(axis=1).std()), 4),
                     "average maximum": round(float(draws.max(axis=1).mean()), 3)})
print(pd.DataFrame(rows).to_string(index=False))

### It barely shrinks, and it is estimating a moving target

On the skewed population the standard error of the mean falls from 1.34 to 0.042 - a factor of 32 -
while the standard error of the maximum goes from 4.47 to 3.58, a factor of 1.25. On the normal
population, 0.32 to 0.010 against 0.59 to 0.30.

And look at the last column: **the average maximum keeps growing** - 13.8, 21.9, 29.7, 37.2. More
data does not make your estimate of "the worst case" more accurate; it makes the observed worst case
*larger*, because you have given the tail more chances.

**What this says about estimating a worst case from a sample:** you cannot, in the way you can
estimate a mean.

- The sample maximum is not converging on anything. For an unbounded distribution there is no
  population maximum for it to converge on, and the sample maximum grows without limit as `n` grows.
- Its uncertainty stays roughly constant, so collecting more data does not buy precision - it buys a
  bigger number.
- Every guarantee you might want to state - "the worst response time is under 2 seconds", "the
  largest claim will be under a million" - is a statement about a tail, and the sample's own extremes
  are the least reliable part of it.

The general lesson, and a fitting end to the chapter: **`1 / sqrt(n)` is a fact about averages, not a
fact about statistics in general.** Quantities that depend on extremes - maxima, 99.9th percentiles,
worst-case latency, tail risk - converge slowly if at all, which is why those quantities are
estimated with distributional assumptions rather than by looking at the sample, and why the
assumptions have to be stated.

## Where to go next

**03-03 · Uncertainty: error bars by resampling.** Everything in this chapter used a population that
does not exist. The next chapter gets standard errors and intervals out of a single sample, with no
formula and no assumed shape, by resampling the data you actually have - which is why the bootstrap
is the most portable idea in this module.